# 27 Temporary Views

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Know the difference between temporary views, global temporary views, permanent views and tables: how to create each, how long it lives, who can see it, and which one to use in a pipeline.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Views are how you name intermediate results, share logic with SQL users, and hide complexity. Choosing the wrong kind causes "table not found" errors in jobs (a temp view from another notebook), stale or missing data, or objects nobody else can query. "Temp view vs global temp view vs table" is a common interview and exam question.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Views and tables</b><br>
A <b>view</b> is a saved <b>query</b>, not saved data. Every time you query it, its query runs again on the current data. A <b>table</b> stores data.
</div>

| Object | Created with | Lives until | Visible to | Stored in catalog? |
|---|---|---|---|---|
| Temporary view | `df.createOrReplaceTempView("v")` / `CREATE TEMP VIEW` | The Spark session ends | The current session only | No |
| Global temporary view | `df.createOrReplaceGlobalTempView("v")` → `global_temp.v` | The Spark application ends | All sessions on the same cluster | No (special `global_temp` schema) |
| View | `CREATE VIEW catalog.schema.v AS ...` | Dropped explicitly | Anyone with permission | Yes (Unity Catalog) |
| Materialized view | `CREATE MATERIALIZED VIEW ...` | Dropped explicitly | Anyone with permission | Yes, and **stores** precomputed results |
| Table | `saveAsTable`, `CREATE TABLE` | Dropped explicitly | Anyone with permission | Yes, stores data |

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A Silver notebook registers <code>orders_clean</code> as a temp view so its SQL cells can use it. That works interactively, but when the notebook is split into two job tasks, the second task fails with <code>TABLE_OR_VIEW_NOT_FOUND</code>, because each task has its own session. The fix is to write <code>orders_clean</code> as a table (or a permanent view on a table) that both tasks can read.
</div>

## Syntax

```python
df.createOrReplaceTempView("orders_v")          # session scope
df.createTempView("orders_v")                    # fails if it already exists
df.createOrReplaceGlobalTempView("orders_g")    # query as global_temp.orders_g

spark.catalog.dropTempView("orders_v")
spark.catalog.listTables()                       # includes temp views (isTemporary=True)
```

```sql
CREATE OR REPLACE TEMP VIEW big_orders AS SELECT * FROM orders WHERE amount > 100;
CREATE OR REPLACE VIEW workspace.spark_basics.big_orders AS SELECT * FROM workspace.spark_basics.orders WHERE amount > 100;
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the schema for permanent objects and an orders DataFrame.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A table prefix such as <code>workspace.spark_basics</code>, then 6 orders.
</div>

In [0]:
CATALOG, SCHEMA = "workspace", "spark_basics"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
DB = f"{CATALOG}.{SCHEMA}"
print("Permanent objects go to", DB)

In [0]:
from pyspark.sql import functions as F

orders = spark.createDataFrame(
    [("O1", "C1", 120.0), ("O2", "C1", 80.0), ("O3", "C2", 300.0), ("O4", "C3", 45.0), ("O5", "C3", 500.0), ("O6", "C4", 60.0)],
    "order_id STRING, customer_id STRING, amount DOUBLE",
)
orders.show()

## 1. Temporary views

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Registers a temp view, queries it from SQL and Python, then shows that <code>createTempView</code> fails if the name already exists.<br><br>
<b>Why it matters</b><br>Temp views are the standard bridge between PySpark and SQL inside one notebook or job task. Use <code>createOrReplaceTempView</code> so re-running a cell doesn't fail.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
SQL and Python both return the 3 orders above 100. The second <code>createTempView</code> fails with a <code>TEMP_TABLE_OR_VIEW_ALREADY_EXISTS</code> error.
</div>

In [0]:
orders.createOrReplaceTempView("orders_v")

spark.sql("SELECT * FROM orders_v WHERE amount > 100").show()
spark.table("orders_v").filter("amount > 100").show()       # spark.table works on views too

try:
    orders.createTempView("orders_v")
except Exception as e:
    print(type(e).__name__, "-", str(e).splitlines()[0][:110])

## 2. A view is a query, not a copy

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates a SQL temp view on top of another view, then replaces the underlying data and queries the view again.<br><br>
<b>Why it matters</b><br>Views don't store data. They always reflect the <b>current</b> definition of what they are built on. Stacking views is a clean way to build logic step by step.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
First the view returns 3 big orders. After <code>orders_v</code> is replaced with data that has 4 big orders, the same <code>big_orders</code> view returns 4 rows without being recreated.
</div>

In [0]:
spark.sql("CREATE OR REPLACE TEMP VIEW big_orders AS SELECT * FROM orders_v WHERE amount > 100")
print("big orders before:", spark.table("big_orders").count())

new_orders = orders.union(spark.createDataFrame([("O7", "C5", 999.0)], orders.schema))
new_orders.createOrReplaceTempView("orders_v")
print("big orders after :", spark.table("big_orders").count())

## 3. Listing and dropping views

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Lists the tables and views visible in the session, then drops a temp view.<br><br>
<b>Why it matters</b><br><code>spark.catalog</code> lets you inspect what exists programmatically, which is handy in utilities and tests. Temp views appear with <code>isTemporary = True</code> and no catalog or schema.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A list including <code>orders_v</code> and <code>big_orders</code> (temporary), then <code>True</code> from <code>dropTempView</code> and the shorter list.
</div>

In [0]:
for t in spark.catalog.listTables():
    if t.isTemporary:
        print(f"{t.name:<12} temporary={t.isTemporary}")

print("dropped:", spark.catalog.dropTempView("big_orders"))
print([t.name for t in spark.catalog.listTables() if t.isTemporary])

## 4. Global temporary views

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tries to create a global temp view and query it through the <code>global_temp</code> schema.<br><br>
<b>Why it matters</b><br>A global temp view is shared by all sessions on the same classic cluster, until the cluster restarts. It lives in the system schema <code>global_temp</code>, so you must always qualify it.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Global temporary views are <b>not supported on serverless compute</b>. Databricks recommends session temp views or tables instead. On Free Edition this cell will most likely print an error. On a classic cluster it returns the rows. Either result is a useful observation.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Either the 6 orders from <code>global_temp.orders_g</code>, or a "not supported" message on serverless.
</div>

In [0]:
try:
    orders.createOrReplaceGlobalTempView("orders_g")
    spark.sql("SELECT count(*) AS n FROM global_temp.orders_g").show()
except Exception as e:
    print("Global temp views not available here:", type(e).__name__, "-", str(e).splitlines()[0][:120])

## 5. Permanent views in Unity Catalog

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Saves the orders as a table, creates a permanent view on it, describes it, and queries it.<br><br>
<b>Why it matters</b><br>Permanent views are stored in Unity Catalog with a three-part name, so other users, jobs, dashboards and Genie can query them, subject to permissions. They are the right way to share business logic. They can only reference permanent objects (tables and views), <b>not</b> temp views.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The view returns the 3 big orders. <code>DESCRIBE EXTENDED</code> shows <code>Type: VIEW</code> and the view's SQL text. Creating a permanent view on a temp view fails.
</div>

In [0]:
orders.write.mode("overwrite").saveAsTable(f"{DB}.lesson27_orders")

spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.lesson27_big_orders
    COMMENT 'Orders above 100'
    AS SELECT order_id, customer_id, amount FROM {DB}.lesson27_orders WHERE amount > 100
""")
spark.table(f"{DB}.lesson27_big_orders").show()

spark.sql(f"DESCRIBE EXTENDED {DB}.lesson27_big_orders").filter("col_name IN ('Type', 'View Text', 'Comment')").show(truncate=False)

try:
    spark.sql(f"CREATE OR REPLACE VIEW {DB}.lesson27_bad AS SELECT * FROM orders_v")
except Exception as e:
    print("Permanent view on a temp view:", type(e).__name__, "-", str(e).splitlines()[0][:110])

## 6. Materialized views (preview)

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Explains materialized views, which store precomputed results and refresh incrementally.<br><br>
<b>Why it matters</b><br>A normal view recomputes on every query. A <b>materialized view</b> stores the result and Databricks refreshes it (incrementally where possible), so dashboards are fast. Materialized views and <b>streaming tables</b> are the Gold- and Silver-layer objects of Lakeflow Declarative Pipelines, and they appear in the exam's transformation domain. They are covered in <code>07_workflows</code>.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
Materialized views need a serverless SQL warehouse or a pipeline to refresh. You can create one in Free Edition from the SQL editor. The statement is shown here for reference.
</div>

```sql
CREATE OR REPLACE MATERIALIZED VIEW workspace.spark_basics.daily_revenue AS
SELECT order_date, sum(amount) AS revenue
FROM workspace.spark_basics.orders
GROUP BY order_date;

REFRESH MATERIALIZED VIEW workspace.spark_basics.daily_revenue;
```

## Under the hood

```
createOrReplaceTempView("v")
   -> stores the DataFrame's LOGICAL PLAN under the name "v" in the session catalog
   -> no data is computed, copied or cached

SELECT ... FROM v
   -> the name is replaced by the stored plan, then the whole query is optimized together
```

<span style="background:#2F6FED;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Transformation</span> creating a temp view runs nothing.

<span style="background:#1E9E5A;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Metadata only</span> a permanent view stores only its SQL text and schema in Unity Catalog.

Because the view's plan is inlined, filters on a view are pushed down into the underlying query just like filters on a DataFrame. The plan below has no "view" node, just the original source with the filter applied.

In [0]:
spark.range(1000).withColumn("amount", F.col("id") * 2).createOrReplaceTempView("numbers_v")
spark.sql("SELECT * FROM numbers_v WHERE amount > 1990").explain()

In [0]:
# Optional cleanup: remove this lesson's permanent objects.
spark.sql(f"DROP VIEW IF EXISTS {DB}.lesson27_big_orders")
spark.sql(f"DROP TABLE IF EXISTS {DB}.lesson27_orders")

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: <code>TABLE_OR_VIEW_NOT_FOUND</code> in a job but the notebook worked</b><br>
The temp view was created in a different session (another notebook, another task, or before a restart). Write a table, or recreate the view in the same task.<br><br>
**⛔ Problem: <code>global_temp</code> view not found**<br>Global temp views must be queried as <code>global_temp.name</code>, and they aren't supported on serverless.<br><br>
<b>⛔ Problem: permanent view creation fails with a temporary object error</b><br>A permanent view can't reference a temp view or a Python DataFrame. Build it on tables or other permanent views.<br><br>
<b>⛔ Problem: a view is slow every time it is queried</b><br>A view recomputes on each query. Use a table or a materialized view for expensive results.<br><br>
**⛔ Problem: <code>TEMP_TABLE_OR_VIEW_ALREADY_EXISTS</code>**<br>Use <code>createOrReplaceTempView</code> / <code>CREATE OR REPLACE TEMP VIEW</code>.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. What is the difference between a temp view and a global temp view?</b><br>
A temp view is visible only in the Spark session that created it and disappears when the session ends. A global temp view is shared across sessions of the same Spark application (cluster), lives in the <code>global_temp</code> schema, and disappears when the application stops.<br><br>

<b>🎤 2. What is the difference between a view and a table?</b><br>
A table stores data. A view stores a query, which runs again every time the view is queried, so it always reflects current data and costs compute on each read.<br><br>

<b>🎤 3. Does creating a temp view cache the data?</b><br>
No. It only registers the logical plan under a name. The query runs when the view is used.<br><br>

<b>🎤 4. Why can a temp view cause a job to fail even though the notebook worked interactively?</b><br>
Each job task has its own Spark session, so a temp view from another task or notebook doesn't exist there. Shared intermediate results should be written as tables.<br><br>

<b>🎤 5. What is a materialized view?</b><br>
A view whose results are precomputed and stored, then refreshed (incrementally when possible). Reads are fast like a table, and the logic stays declarative like a view. On Databricks they are managed by Lakeflow Declarative Pipelines or SQL warehouses.<br><br>

<b>🎤 6. Can a permanent view reference a temp view?</b><br>
No. A permanent view is stored in the catalog and can be queried by others, so it can only depend on permanent objects.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. A data engineer creates a temporary view in notebook A. A job runs notebook B as a separate task and queries the view. What happens?</b><br>
A. Notebook B reads the view normally<br>
B. Notebook B fails because the view only exists in notebook A's session<br>
C. Notebook B reads a cached copy<br>
D. The view is automatically promoted to a table<br>
<details><summary><b>Show answer</b></summary><b>B.</b> Temporary views are session-scoped.</details><br><br>

<b>Q2. Which object stores precomputed query results that are refreshed incrementally on Databricks?</b><br>
A. Temporary view<br>
B. View<br>
C. Materialized view<br>
D. Global temporary view<br>
<details><summary><b>Show answer</b></summary><b>C.</b></details><br><br>

<b>Q3. How do you query a global temporary view named <code>daily</code>?</b><br>
A. <code>SELECT * FROM daily</code><br>
B. <code>SELECT * FROM temp.daily</code><br>
C. <code>SELECT * FROM global_temp.daily</code><br>
D. <code>SELECT * FROM default.daily</code><br>
<details><summary><b>Show answer</b></summary><b>C.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Create a temp view <code>customer_totals</code> with each customer's total order amount, using SQL <code>CREATE OR REPLACE TEMP VIEW</code></li><li>Create a second temp view <code>top_customers</code> on top of it with the customers whose total is above 200</li><li>Change the data behind <code>customer_totals</code> and show that <code>top_customers</code> reflects the change without being recreated</li><li>Create a permanent view <code>lesson27_top_customers</code> in your schema on top of the <code>lesson27_orders</code> table</li><li>In one sentence: when would you use a table instead of a permanent view?</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1-2. Stacked temp views
orders.createOrReplaceTempView("orders_v")
spark.sql("CREATE OR REPLACE TEMP VIEW customer_totals AS SELECT customer_id, sum(amount) AS total FROM orders_v GROUP BY customer_id")
spark.sql("CREATE OR REPLACE TEMP VIEW top_customers AS SELECT * FROM customer_totals WHERE total > 200")
spark.table("top_customers").show()

# 3. Change the base data: C4 now has a big order
orders.union(spark.createDataFrame([("O8", "C4", 400.0)], orders.schema)).createOrReplaceTempView("orders_v")
spark.table("top_customers").show()

# 4. Permanent view on a permanent table
orders.write.mode("overwrite").saveAsTable(f"{DB}.lesson27_orders")
spark.sql(f"""
    CREATE OR REPLACE VIEW {DB}.lesson27_top_customers AS
    SELECT customer_id, sum(amount) AS total FROM {DB}.lesson27_orders GROUP BY customer_id HAVING sum(amount) > 200
""")
spark.table(f"{DB}.lesson27_top_customers").show()

# 5. Use a table when the result is expensive to compute and read often, or must be a stable snapshot.

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>A view stores a query. A table stores data</li><li>Temp view: current session only. Global temp view: all sessions on a cluster, <code>global_temp.</code> prefix, not on serverless</li><li>Permanent views live in Unity Catalog, are shareable and permission-controlled, and can't use temp objects</li><li>Materialized views store precomputed results and refresh incrementally</li><li>Creating a temp view runs nothing. Its plan is inlined and optimized with the query</li><li>Temp views don't survive across job tasks. Write tables to share data between tasks</li></ul>
</div>

| Task | Code |
|---|---|
| Temp view | `df.createOrReplaceTempView("v")` |
| Temp view in SQL | `CREATE OR REPLACE TEMP VIEW v AS SELECT ...` |
| Global temp view | `df.createOrReplaceGlobalTempView("g")` → `global_temp.g` |
| Permanent view | `CREATE OR REPLACE VIEW cat.sch.v AS SELECT ...` |
| Drop | `spark.catalog.dropTempView("v")`, `DROP VIEW cat.sch.v` |
| List | `spark.catalog.listTables()` |

## Git commit

```
git add 01_spark_basics/27_temporary_views.ipynb
git commit -m "add 27 temporary views notebook"
```